# Exploring QUT-DV25

What does the data actually contain, and can it support the claims we want to make?

Most of the heavy lifting lives in `src/pkgtriage/data/` so the numbers here match what `scripts/run_data_audit.py` writes to `reports/tables/`. Run `python scripts/download_data.py` first.

In [1]:
import pandas as pd
from IPython.display import Image, display

from pkgtriage import plots
from pkgtriage.data.audit import run_audit
from pkgtriage.data.download import load_raw
from pkgtriage.data.schema import COUNT_COLUMNS
from pkgtriage.data.validate import validate_raw

pd.set_option("display.max_colwidth", 80)
df = load_raw()
print(validate_raw(df).summary())

rows: 14270, columns: 38
class counts: {0: 7143, 1: 7127}
duplicate package names: 0
rows identical apart from the name: 48 (0 with conflicting labels)
columns with missing values: 18


## What one row looks like

One row per package. The count columns are easy. The text columns hold a TCP state dictionary, top-N process lists, dependency lists and syscall 3-grams, and all of these need parsing in Phase 4.

In [2]:
df.sample(2, random_state=7).T

,9583,13702
Package_Name,replaceall,websokcets-0.1.tar.gz
Root_DIR_Installation,497,1263
Temporary_DIR_Installation,27,131
Home_DIR_Installation,603,4063
User_Access,1421,2896
Sys_Access,366,35
Etc_DIR_Installation,539,668
Other_DIR_Installation,28462,12416
State_Transition,"{'SYN_SENT -> ->': 146, 'FIN_WAIT1 -> ->': 139, 'FIN_WAIT2 -> ->': 134, 'CLO...","{'CLOSE -> ->': 72, 'SYN_SENT -> ->': 72, 'ESTABLISHED -> ->': 72, 'FIN_WAIT..."
Local_IP_Address_Access,2,2


## Label and how the packages were installed

`Level` is the label (1 = malicious). The package name gives away how each sample was installed: malicious ones are archive file names, benign ones are plain PyPI names. The raw pip logs confirm it (`Processing ./x.tar.gz` versus `Downloading ...`). So `Package_Name` can never be a feature, and anything that depends on installing from PyPI is suspect.

In [3]:
audit = run_audit(df)
audit["name_format_by_class"]

Level,benign,malicious
name format,,
archive file name,0,7125
plain name,7143,2


## Missing values are not random

`Direct_Dependencies` is empty for almost every malicious package, which is what you would expect when installing from a local archive without pulling metadata from PyPI. Missingness alone separates the classes with an AUC of about 0.75.

In [4]:
display(audit["missingness_by_class"].round(2))
plots.use_report_style()
fig = plots.missingness_figure(audit["missingness_by_class"])

,column,missing_benign_pct,missing_malicious_pct,auc_of_missingness
6,Direct_Dependencies,49.64,99.99,0.75
7,Indirect_Dependencies,64.24,89.29,0.63
15,Pattern_8,0.31,7.30,0.53
16,Pattern_9,0.31,7.30,0.53
17,Pattern_10,0.31,7.30,0.53
12,Pattern_5,0.31,7.30,0.53
11,Pattern_4,0.31,7.27,0.53
10,Pattern_3,0.31,7.27,0.53
9,Pattern_2,0.27,7.14,0.53
13,Pattern_6,0.31,7.14,0.53


## Count features

Malicious installs touch the home, root and /etc directories more. But benign installs make *more* network and process calls, which runs against the usual story that malware phones home. A simpler explanation is that benign packages downloaded their dependencies from PyPI.

In [5]:
fig = plots.count_distributions_figure(df, ["Network_Operations", "Home_DIR_Installation",
    "Process_Management_Operations", "Root_DIR_Installation", "Etc_DIR_Installation", "Remote_IP_Address_Access"])

In [6]:
audit["single_feature_auc"].round(3)

,feature,auc,median_benign,median_malicious
12,Network_Operations,0.806,115.0,41.0
2,Home_DIR_Installation,0.795,675.0,2707.0
13,Process_Management_Operations,0.784,63.0,29.0
3,User_Access,0.753,693.0,1951.0
5,Etc_DIR_Installation,0.736,184.0,540.0
0,Root_DIR_Installation,0.726,0.0,497.0
6,Other_DIR_Installation,0.686,8161.0,28358.0
8,Remote_IP_Address_Access,0.684,48.0,23.0
14,IO_Operations,0.666,542.0,313.0
10,Remote_Port_Access,0.666,18.0,10.0


No single count is decisive (best AUC is about 0.81), so there's no one-column leak among the counts. The risk is in combinations, together with the artefacts below.

In [7]:
fig = plots.correlation_figure(df, COUNT_COLUMNS)

## Background processes

The process lists are machine-wide top-N lists, so they mostly name the tracing tools and desktop services. Some of these are close to class-exclusive: `tracker-miner-f` appears only in malicious runs, `ThreadPoolForeg` (a browser thread) and `pip3` almost only in benign ones. That points to the classes being recorded in different sessions, and it means these tokens must be dropped before modelling.

In [8]:
tok = audit["process_tokens_by_class"]
display(tok.head(15).round(1))
fig = plots.process_tokens_figure(tok)

,token,benign_pct,malicious_pct,gap_pct,group
0,sqldb:c~lite,39.9,69.7,29.7,desktop
1,tracker-miner-f,0.0,27.8,27.8,desktop
2,clear,34.8,11.6,23.1,desktop
3,systemd-oomd,17.5,2.3,15.2,desktop
4,ThreadPoolForeg,15.2,0.0,15.2,desktop
5,pip3,14.4,0.2,14.2,installer
6,StreamTrans,56.8,70.5,13.7,desktop
7,python,15.6,2.1,13.5,installer
8,gnome-system-mo,71.3,82.8,11.5,desktop
9,pool-tracker-mi,0.1,11.6,11.5,desktop


In [9]:
# tokens that plausibly come from the package itself
tok[tok.group == "package"].head(15).round(2)

,token,benign_pct,malicious_pct,gap_pct,group
13,cc1,6.93,0.08,6.85,package
14,single,0.00,5.37,5.37,package
15,rootkit,0.00,4.17,4.17,package
17,cc1plus,3.74,0.00,3.74,package
20,ld,3.30,0.00,3.30,package
25,configmgrWriter,0.00,2.36,2.36,package
27,apt-get,3.02,0.86,2.17,package
28,as,2.04,0.00,2.04,package
33,grep,6.10,7.97,1.87,package
35,Indexed~,0.22,1.52,1.29,package


## Near-duplicate families

Several attackers uploaded dozens of variants of the same package. A random split would scatter them across train and test, so Phase 4 uses a family-grouped split instead.

In [10]:
display(audit["family_stats"].T)
display(audit["family_sizes"].head(10))
fig = plots.family_size_figure(audit["family_sizes"])

,0
families,12960
families_with_variants,585
rows_in_multi_member_families,1895
largest_family,bytedtrace
largest_family_size,41
mixed_label_families,60


,size,malicious_share
family,,
bytedtrace,41,1.000000
xcryptography,30,1.000000
builderknower,24,1.000000
important-package,24,1.000000
http3-client,22,1.000000
pptest,21,1.000000
sentinelone,21,1.000000
aiohttp-proxies,20,1.000000
pycrypterexe,19,1.000000


## What this means for the project

1. Drop `Package_Name` and every environment process token.
2. Treat dependency and network features as possibly reflecting install method; measure their effect with ablations in Phase 5.
3. Use a family-grouped split.
4. Report results with those caveats rather than a headline accuracy.

Figures for the report are written to `images/` by `scripts/run_data_audit.py`.